In [1]:
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --progress-bar off
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install -qqq --no-deps {xformers} trl peft accelerate bitsandbytes triton --progress-bar off

import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [2]:
##Load Model
max_seq_len = 2048
model,tokenizer=FastLanguageModel.from_pretrained(model_name='unsloth/Meta-Llama-3.1-8B-bnb-4bit',
                                                  max_seq_length=max_seq_len,
                                                  load_in_4bit=True,
                                                  dtype=None)

==((====))==  Unsloth 2024.9: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.4.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaExtendedRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,),

In [4]:
##Prepare model for PEFT
model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=16,lora_dropout=0,
                                       target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
                                       use_rslora=True,
                                       use_gradient_checkpointing='unsloth')

Unsloth 2024.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
tokenizers=get_chat_template(tokenizer,
                            chat_template='chatml',
                            mapping={"role":"from","content":"value","user":"human","assistant":"gpt"})

Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
dataset=load_dataset("mlabonne/FineTome-100k",split="train[:200]")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [14]:
# Assuming `get_chat_template` properly configures the tokenizer for your chat format
tokenizer = get_chat_template(
    tokenizer,
    chat_template='chatml',
    mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"}
)

# Define a function to apply the template correctly and verify its output
def apply_template(examples):
    messages = examples['conversations']
    formatted_text = []

    for message in messages:
        # Apply the chat template to format the text
        formatted = tokenizer.apply_chat_template(
            message,
            tokenizer=False,  # Ensure this setting is correct for formatting, not tokenization
            add_generation_prompt=False
        )

        # Check if formatted output is in token format
        if isinstance(formatted, list) and all(isinstance(x, int) for x in formatted):
            # Decode tokens if needed (if the output is token IDs)
            decoded_text = tokenizer.decode(formatted, skip_special_tokens=True)
            formatted_text.append(decoded_text)
        else:
            # If the output is already text, just append it
            formatted_text.append(formatted)

    return {"text": formatted_text}

# Map the function to the dataset
dataset = dataset.map(apply_template, batched=True)

# Print the formatted text of the first item
print(f"Formatted Output: {dataset['text'][0]}")


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Formatted Output: <|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. 

Furthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.

Finally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.
<|im_start|>assistant
Boolean operato

In [31]:
# Print the formatted text of the first item
print(f"Formatted Output: {dataset['text'][2]}")

Formatted Output: <|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions.

Furthermore, discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. Finally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions.
<|im_start|>assistant
Boolean operators are logical operators used to combine or manipulate boolean values in programming. They allow you to perform comparisons and create complex logical expressions. The three main boolean operators are:

1. AND operator (&&): Returns true if both operands are true. Otherwise, it returns false. For example:
   - `true && true` returns true
   - `true && false` returns false

2.

In [15]:
trainer=SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=max_seq_len,
    dataset_num_proc=2,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        fp16= not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        warmup_steps=10,
        output_dir='output',
        seed=0
    ),
)


Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

In [16]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 200 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 4
\        /    Total batch size = 16 | Total steps = 12
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,1.164800
2,1.081800
3,1.211200
4,0.954200
5,0.955600
6,0.909400
7,0.914700
8,0.789500
9,0.781400
10,0.742200


TrainOutput(global_step=12, training_loss=0.924407109618187, metrics={'train_runtime': 483.0126, 'train_samples_per_second': 0.414, 'train_steps_per_second': 0.025, 'total_flos': 8106060421988352.0, 'train_loss': 0.924407109618187, 'epoch': 0.96})

In [17]:
model=FastLanguageModel.for_inference(model)
messages=[
    {'from':'human','value':'Is 9.11 larger than 9.9'}]
inputs=tokenizer.apply_chat_template(messages,tokenizer=False,add_generation_prompt=True,return_tensors='pt').to('cuda')


In [18]:
text_streamer=TextStreamer(tokenizer)
_=model.generate(inputs=inputs,max_new_tokens=100,streamer=text_streamer,use_cache=True)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
Is 9.11 larger than 9.9<|im_end|>
<|im_start|>assistant
Yes, 9.11 is larger than 9.9. 9.11 is a decimal number that is greater than 9.9. When comparing decimal numbers, we compare their fractional parts. In this case, 9.11 has a fractional part that is larger than 9.9's fractional part, which means 9.11 is larger than 9.9.
9.11 can be written as 9 + 0.11, where 0.11


In [19]:
model.save_pretrained_merged('model',tokenizer,save_method='merged_16bit')


Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which will take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 2.75 out of 12.67 RAM for saving.


100%|██████████| 32/32 [04:30<00:00,  8.45s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving model... This might take 5 minutes for Llama-7b...
Unsloth: Saving model/pytorch_model-00001-of-00004.bin...
Unsloth: Saving model/pytorch_model-00002-of-00004.bin...
Unsloth: Saving model/pytorch_model-00003-of-00004.bin...
Unsloth: Saving model/pytorch_model-00004-of-00004.bin...
Done.


In [27]:
from huggingface_hub import notebook_login

# This will prompt you to enter your Hugging Face token
notebook_login()


In [29]:
# Replace 'your-username' with your actual Hugging Face username or organization name
username = "Kartik12"  # Update this with your Hugging Face username
repo_name_1 = f"{username}/Meta-Llama-3.1-8B"
repo_name_2 = f"{username}/Meta-Llama-3.1-8B-gguf"

# Push the model to the first repo
try:
    model.push_to_hub(repo_name_1, tokenizer=tokenizer, save_method='merged_16bit')
    print(f"Model successfully pushed to: {repo_name_1}")
except Exception as e:
    print(f"Error pushing model to {repo_name_1}: {e}")

# Push the model to the second repo
try:
    model.push_to_hub(repo_name_2, tokenizer=tokenizer, save_method='merged_16bit')
    print(f"Model successfully pushed to: {repo_name_2}")
except Exception as e:
    print(f"Error pushing model to {repo_name_2}: {e}")


README.md:   0%|          | 0.00/589 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Kartik12/Meta-Llama-3.1-8B
Model successfully pushed to: Kartik12/Meta-Llama-3.1-8B


README.md:   0%|          | 0.00/589 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Kartik12/Meta-Llama-3.1-8B-gguf
Model successfully pushed to: Kartik12/Meta-Llama-3.1-8B-gguf
